### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [1]:
%pip install numpy scikit-learn

  Using cached scikit_learn-1.8.0-cp314-cp314-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 15.7 MB/s  0:00:00eta 0:00:01
Using cached scikit_learn-1.8.0-cp314-cp314-macosx_12_0_arm64.whl (8.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 18.5 MB/s  0:00:01 eta 0:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [3]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [4]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [6]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [7]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [8]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [9]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [10]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [11]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [12]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [13]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [14]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [15]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [16]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [17]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [18]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  1534, 10055,  4750], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [19]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [20]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [21]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [22]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [23]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [24]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


In [ ]:
# TÚ CODIGO AQUÍ

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Elegimos 5 índices de documentos al azar
# Usamos seed(42) para que "al azar" sea siempre igual 
np.random.seed(42) 
random_indices = np.random.choice(X_train.shape[0], 5, replace=False)

for doc_idx in random_indices:
    print("=" * 70)
    print(f"DOCUMENTO ELEGIDO (Índice {doc_idx}) - Categoría: {newsgroups_train.target_names[y_train[doc_idx]]}")
    print("-" * 30)
    # Mostramos los primeros 300 caracteres del documento para no llenar la pantalla
    print(newsgroups_train.data[doc_idx][:300] + "...") 
    
    # 1. Medimos la coincidencia del documento con TODOS los documentos
    cossim = cosine_similarity(X_train[doc_idx], X_train)[0]
    
    # 2. Obtenemos el "Top 5". El más similar es él mismo (posición 0), así que tomamos del 1 al 6.
    mostsim = np.argsort(cossim)[::-1][1:6]
    
    print("\n--- SUS 5 DOCUMENTOS MÁS PARECIDOS ---")
    for i, sim_idx in enumerate(mostsim):
        simil_score = cossim[sim_idx]
        simil_class = newsgroups_train.target_names[y_train[sim_idx]]
        # Imprimimos la posición, la similitud, a qué categoría pertenece el texto gemelo, y un pedacito de él
        print(f"\n{i+1}. Índice {sim_idx} | Similitud: {simil_score:.4f} | Categoría: {simil_class}")
        print(newsgroups_train.data[sim_idx][:150].replace('\n', ' ') + "...")


DOCUMENTO ELEGIDO (Índice 7492) - Categoría: comp.sys.mac.hardware
------------------------------
Could someone please post any info on these systems.

Thanks.
BoB
-- 
---------------------------------------------------------------------- 
Robert Novitskey | "Pursuing women is similar to banging one's head
rrn@po.cwru.edu  |  against a wall...with less opportunity for reward" ...

--- SUS 5 DOCUMENTOS MÁS PARECIDOS ---

1. Índice 10935 | Similitud: 0.6665 | Categoría: comp.sys.mac.hardware
Hey everybody:     I want to buy a mac and I want to get a good price...who doesn't?  So, could anyone out there who has found a really good deal on a...

2. Índice 7258 | Similitud: 0.3476 | Categoría: comp.sys.ibm.pc.hardware
Hay all:      Has anyone out there heard of any performance stats on the fabled p24t.  I was wondering what it's performance compared to the 486/66 an...

3. Índice 4971 | Similitud: 0.1799 | Categoría: comp.sys.mac.hardware
Could someone please send instructions for installin

Segun podemos ver, el documento elegido es como un mail que se refiere a sistemas.

Despues en los elegidos, la similitud es muy baja ya que abordan tópicos muy diferentes, el mas parecido es el 10935 que habla de una mac, y puede de alguna manera estar relacionada a "sistemas".

In [ ]:
# 1. Calculamos a qué texto viejo se parece cada texto NUEVO del examen.
similarities_matrix = cosine_similarity(X_test, X_train)

# 2. Para cada texto de test, buscamos el texto de train que tuvo mayor puntuación (argmax).
most_similar_train_indices = np.argmax(similarities_matrix, axis=1)

# 3. Le asignamos al texto nuevo la categoría que tenga su "gemelo" viejo.
y_pred_zeroshot = y_train[most_similar_train_indices]

# 4. Evaluamos cómo le fue en el examen a la computadora usando esta estrategia
f1_zeroshot = f1_score(y_test, y_pred_zeroshot, average='macro')

print(f"F1-Score del modelo por similitudes (Zero-Shot): {f1_zeroshot:.4f}")


F1-Score del modelo por similitudes (Zero-Shot): 0.5050


El score es 0.5, lo cual es un buen score para un modelo no entrenado. Tomando en cuenta que el modelo lo que está haciendo es buscar una palabra mas cercana y en base a eso buscar similitudes es un buen numero.


In [ ]:
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Afinamos el vectorizador 
# max_df=0.5: Si una palabra aparece en más de la mitad de los textos, la ignoramos.
# min_df=5: Si una palabra aparece en menos de 5 textos, no sirve de referencia, se ignora.
tfidf_mejorado = TfidfVectorizer(max_df=0.5, min_df=5)

# Convertimos los textos usando esta nueva mejora
X_train_mejorado = tfidf_mejorado.fit_transform(newsgroups_train.data)
X_test_mejorado = tfidf_mejorado.transform(newsgroups_test.data)

print(f"El vocabulario ahora tiene menos palabras basura: {X_train_mejorado.shape[1]} palabras únicas.\n")

# 2. ENTRENAMOS Y EXAMINAMOS MULTINOMIAL NAÏVE BAYES (Mejorado)
# alpha=0.1 suaviza las probabilidades, normalmente funciona bien
clf_multinomial = MultinomialNB(alpha=0.1)
clf_multinomial.fit(X_train_mejorado, y_train)
y_pred_multi = clf_multinomial.predict(X_test_mejorado)
print(f"Nota (F1-Score) de MultinomialNB mejorado: {f1_score(y_test, y_pred_multi, average='macro'):.4f}")

# 3. ENTRENAMOS Y EXAMINAMOS COMPLEMENT NAÏVE BAYES
clf_complement = ComplementNB(alpha=0.1)
clf_complement.fit(X_train_mejorado, y_train)
y_pred_comp = clf_complement.predict(X_test_mejorado)
print(f"Nota (F1-Score) de ComplementNB mejorado:  {f1_score(y_test, y_pred_comp, average='macro'):.4f}")


El vocabulario ahora tiene menos palabras basura: 18092 palabras únicas.

Nota (F1-Score) de MultinomialNB mejorado: 0.6742
Nota (F1-Score) de ComplementNB mejorado:  0.6764


Pasamos de 130000 palabras a 18092, lo cual es muy bueno ya que eliminamos palabras basura como "por", "a", "y", y palabrs que se repetían constantemente en los documentos que constituian basura.
También eliminamos palabras con poca valía, donde si una palabra aparecía en menos de 5 textos, la eliminabamos tambien, ya que no nos aporta mucho a nuestro modelo.


El f1-score subió debido a la limpieza de palabras.

Se observa un rendimiento levemente mejor en el ComplementNB.

In [31]:
# 1. Giramos (transponemos) nuestra vieja tabla
X_train_transposed = X_train.T 

print(f"Antes teníamos: {X_train.shape} (Textos, Palabras)")
print(f"Ahora girada es: {X_train_transposed.shape} (Palabras, Textos)\n")

# 2. Elegimos 5 palabras claras que sepamos que están en el diccionario
palabras_elegidas = ["windows", "baseball", "god", "morning", "doctor"]

for palabra in palabras_elegidas:
    print("=" * 50)
    print(f"PALABRA ELEGIDA: '{palabra}'")
    
    # 3. Buscamos el código numérico de esta palabra
    palabra_idx = tfidfvect.vocabulary_.get(palabra)
    
    if palabra_idx is None:
        print(f"Error: La palabra '{palabra}' no se usó en los textos.")
        continue
        
    # Extraemos el vector que le pertenece a esa palabra 
    vector_palabra = X_train_transposed[palabra_idx]
    
    # La comparamos midiendo similitud, pero contra las demás PALABRAS
    cossim_palabras = cosine_similarity(vector_palabra, X_train_transposed)[0]
    
    # Las 5 palabras con la puntuación más alta
    mostsim_idx = np.argsort(cossim_palabras)[::-1][1:6]
    
    print("Top 5 de palabras más amigas/similares:")
    for i in mostsim_idx:
        # Obtenemos la palabra usando el índice, como tenías armado en el código original
        palabra_similar = idx2word[i]
        similitud = cossim_palabras[i]
        print(f" - {palabra_similar} (Coincidencia: {similitud:.4f})")


Antes teníamos: (11314, 101631) (Textos, Palabras)
Ahora girada es: (101631, 11314) (Palabras, Textos)

PALABRA ELEGIDA: 'windows'
Top 5 de palabras más amigas/similares:
 - dos (Coincidencia: 0.3037)
 - ms (Coincidencia: 0.2320)
 - microsoft (Coincidencia: 0.2219)
 - nt (Coincidencia: 0.2140)
 - for (Coincidencia: 0.1930)
PALABRA ELEGIDA: 'baseball'
Top 5 de palabras más amigas/similares:
 - tommorrow (Coincidencia: 0.1839)
 - football (Coincidencia: 0.1759)
 - penna (Coincidencia: 0.1734)
 - wintry (Coincidencia: 0.1690)
 - espn (Coincidencia: 0.1677)
PALABRA ELEGIDA: 'god'
Top 5 de palabras más amigas/similares:
 - jesus (Coincidencia: 0.2688)
 - bible (Coincidencia: 0.2616)
 - that (Coincidencia: 0.2560)
 - existence (Coincidencia: 0.2548)
 - christ (Coincidencia: 0.2511)
PALABRA ELEGIDA: 'morning'
Top 5 de palabras más amigas/similares:
 - inaccuracy (Coincidencia: 0.3026)
 - turnpike (Coincidencia: 0.2593)
 - emanating (Coincidencia: 0.1949)
 - environet (Coincidencia: 0.1948)
 -

Se puede ver que la transposición de la matriz arroja similitudes precisas cuando se trata de palabras fuertes como baseball, god, doctor, y especialmente windows, donde el modelo logró extraer sus asociaciones informáticas de los años 90 (Microsoft, NT, MS-DOS) sin contexto previo. Por otro lado, para palabras genéricas o de uso diario como morning, las coincidencias están totalmente alejadas de la realidad. Esto se debe a que son términos ambiguos.